# 🚀 Build Vector Index RAG PUSDIKZI (100% Standalone - Google Colab GPU)

Notebook ini **100% Mandiri (Standalone)** untuk dijalankan langsung di **VM Runtime GPU Google Colab** tanpa memerlukan folder `config/`, `src/`, maupun `scripts/` di server Colab:
- **Sel 1**: Cek GPU NVIDIA CUDA (`nvidia-smi` & `torch.cuda.is_available()`) dan instalasi library.
- **Sel 2**: Konfigurasi parameter `.env` untuk GPU Google Colab (`device='cuda'`, `batch_size=64`).
- **Sel 3**: Definisi mandiri modul `DocumentPage`, `TextCleaner`, & `TextChunker` beserta bundel otomatis 16 dokumen dari `data/raw/SECRET`.
- **Sel 4**: Definisi mandiri `EmbeddingEngine` (proses kalkulasi vektor menggunakan model `nomic-ai/nomic-embed-text-v1.5` di GPU CUDA) & `ChromaVectorStore`.
- **Sel 5**: Menjalankan pipeline `run_indexing()` pada dokumen `data/raw/SECRET` dengan GPU Colab.
- **Sel 6 & 7**: Ekspor/unduh hasil vektor dari Colab ke laptop dan impor ke Docker ChromaDB lokal.

In [17]:
# 1. Cek Runtime GPU Google Colab & Instalasi Dependensi
import os
import sys
import subprocess

!pip install -q chromadb>=0.4.24 sentence-transformers>=2.3.0 einops>=0.7.0 \
    pypdf>=4.0.0 pymupdf>=1.23.0 python-docx>=1.1.0 openpyxl>=3.1.0 \
    xlrd>=2.0.1 olefile>=0.46 pyyaml>=6.0 python-dotenv>=1.0.0

import torch
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
print(f"Lingkungan Eksekusi : {'Google Colab VM Runtime' if IN_COLAB else 'Local / Remote Kernel'}")
print(f"PyTorch Version     : {torch.__version__}")
print(f"CUDA Tersedia       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Nama GPU            : {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Kapasitas VRAM GPU  : {vram_gb:.2f} GB")
else:
    print("⚠️ PERINGATAN: GPU CUDA belum terdeteksi. Pastikan memilih Runtime GPU (T4) di Google Colab.")


Lingkungan Eksekusi : Google Colab VM Runtime
PyTorch Version     : 2.11.0+cu130
CUDA Tersedia       : True
Nama GPU            : Tesla T4
Kapasitas VRAM GPU  : 14.56 GB


In [18]:
# 2. Konfigurasi Parameter & Environment untuk Runtime GPU Google Colab
import os
from pathlib import Path
import torch

WORKSPACE_DIR = Path.cwd()
os.environ["RUNTIME_ENV"] = "colab_gpu"
os.environ["EMBEDDING_MODEL_NAME"] = "nomic-ai/nomic-embed-text-v1.5"
os.environ["EMBEDDING_DEVICE"] = "cuda" if torch.cuda.is_available() else "cpu"
os.environ["EMBEDDING_BATCH_SIZE"] = "64"
os.environ["HF_HUB_OFFLINE"] = "0"
os.environ["TRANSFORMERS_OFFLINE"] = "0"
os.environ["CHUNK_SIZE"] = "256"
os.environ["CHUNK_OVERLAP"] = "26"
os.environ["DATA_RAW_DIR"] = str(WORKSPACE_DIR / "data" / "raw" / "SECRET")
os.environ["CHROMA_COLLECTION_NAME"] = "pusdikzi_knowledge_base"
os.environ["CHROMA_PERSIST_DIR"] = str(WORKSPACE_DIR / "data" / "chroma_colab")

SETTINGS = {
    "chromadb": {
        "host": "localhost",
        "port": 8000,
        "collection_name": os.environ["CHROMA_COLLECTION_NAME"],
        "distance_function": "cosine",
        "persist_directory": os.environ["CHROMA_PERSIST_DIR"],
    },
    "embedding": {
        "model_name": os.environ["EMBEDDING_MODEL_NAME"],
        "query_prefix": "search_query: ",
        "document_prefix": "search_document: ",
        "batch_size": int(os.environ["EMBEDDING_BATCH_SIZE"]),
        "device": os.environ["EMBEDDING_DEVICE"],
        "normalize_embeddings": True,
    },
    "ingestion": {
        "supported_extensions": [".pdf", ".txt", ".md", ".docx", ".doc", ".xlsx", ".xls"],
        "data_raw_dir": os.environ["DATA_RAW_DIR"],
        "chunking": {
            "chunk_size": int(os.environ["CHUNK_SIZE"]),
            "chunk_overlap": int(os.environ["CHUNK_OVERLAP"]),
            "overlap_ratio": 0.10,
        },
        "sanitization": {
            "enable_masking": True,
            "mask_patterns": {
                "nik": r"\b[0-9]{16}\b",
                "nrp": r"\b(NRP|NIP)[:\s]*([0-9]{6,18})\b",
                "phone": r"\b(\+62|62|0)8[1-9][0-9]{7,10}\b",
            },
        },
    },
    "security": {"default_classification": "SECRET"},
}

print("=== KONFIGURASI AKTIF ===")
print(f"Model Embedding : {SETTINGS['embedding']['model_name']}")
print(f"Device          : {SETTINGS['embedding']['device']}")
print(f"Batch Size      : {SETTINGS['embedding']['batch_size']}")
print(f"Target Data Dir : {SETTINGS['ingestion']['data_raw_dir']}")
print(f"ChromaDB Dir    : {SETTINGS['chromadb']['persist_directory']}")


=== KONFIGURASI AKTIF ===
Model Embedding : nomic-ai/nomic-embed-text-v1.5
Device          : cuda
Batch Size      : 64
Target Data Dir : /content/data/raw/SECRET
ChromaDB Dir    : /content/data/chroma_colab


In [19]:
# 3. Definisi Mandiri Modul Ingestion, TextCleaner, & TextChunker + Bundel Dokumen SECRET
# (Tidak memerlukan folder config/ atau src/ di VM Google Colab)
import re
import json
import lzma
import base64
import hashlib
import logging
import unicodedata
from pathlib import Path
from typing import List, Dict, Any, Optional

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s - %(message)s")
logger = logging.getLogger("build_index_colab")

class DocumentPage:
    def __init__(self, content: str, page_number: int, metadata: Dict[str, Any]):
        self.content = content
        self.page_number = page_number
        self.metadata = metadata

class DocumentChunk:
    def __init__(self, chunk_id: str, text: str, metadata: Dict[str, Any]):
        self.chunk_id = chunk_id
        self.text = text
        self.metadata = metadata

class TextCleaner:
    def __init__(self):
        san_cfg = SETTINGS["ingestion"]["sanitization"]
        self.enable_masking = san_cfg.get("enable_masking", True)
        self.patterns = san_cfg.get("mask_patterns", {})

    def sanitize(self, text: str) -> str:
        if not text:
            return ""
        text = unicodedata.normalize("NFKC", text)
        text = re.sub(r"[\u200B-\u200F\uFEFF\u2060\u00AD\uFFFD]", "", text)
        text = re.sub(r"[\u00A0\u1680\u2000-\u200A\u202F\u205F\u3000]", " ", text)
        text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\t", " ")
        text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]", "", text)
        text = re.sub(r"[ÿ\x80-\xff]{2,}", "", text)
        text = re.sub(r"[^\S\n]+", " ", text)
        text = re.sub(r"([A-Za-z]{2,})-\n\s*([a-z]{2,})", r"\1\2", text)
        text = re.sub(r"(?im)^[\t ]*(?:page|halaman|hal\.?)*\s*\d+(?:\s*(?:of|dari|\/|-)\s*\d+)?[\t ]*$", "", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        if self.enable_masking:
            text = re.sub(self.patterns.get("nik", r"\b[0-9]{16}\b"), "[NIK-TERPROTEKSI]", text)
            text = re.sub(self.patterns.get("phone", r"\b(\+62|62|0)8[1-9][0-9]{7,10}\b"), "[KONTAK-TERPROTEKSI]", text)
        return text.strip()

class TokenizerHelper:
    @classmethod
    def count_tokens(cls, text: str) -> int:
        if not text:
            return 0
        words_and_symbols = re.findall(r"\w+|[^\w\s]", text)
        return max(1, int(len(words_and_symbols) * 1.15))

class TextChunker:
    def __init__(self, chunk_size: int = 256, chunk_overlap: int = 26):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.cleaner = TextCleaner()
        self.tokenizer = TokenizerHelper()

    def _split_into_structural_units(self, text: str) -> List[Dict[str, Any]]:
        heading_pattern = re.compile(
            r"^(?:#{1,6}\s+.+|BAB\s+[IVXLCDM\d]+[^\n]*|PASAL\s+\d+[^\n]*|BAGIAN\s+[^\n]+|[0-9A-Z]\.\s+[A-Z\s]{3,})$",
            re.IGNORECASE | re.MULTILINE
        )
        units: List[Dict[str, Any]] = []
        for block in text.split("\n\n"):
            block = block.strip()
            if not block:
                continue
            if block.startswith("|") and "\n|" in block:
                units.append({"type": "table", "text": block})
                continue
            lines = block.splitlines()
            if len(lines) == 1 and heading_pattern.match(lines[0]):
                units.append({"type": "heading", "text": lines[0]})
                continue
            if self.tokenizer.count_tokens(block) <= self.chunk_size:
                units.append({"type": "paragraph", "text": block})
            else:
                for s in re.split(r"(?<=[.?!;\n])\s+", block):
                    s = s.strip()
                    if not s:
                        continue
                    if self.tokenizer.count_tokens(s) <= self.chunk_size:
                        units.append({"type": "sentence", "text": s})
                    else:
                        curr_words = []
                        for w in s.split():
                            curr_words.append(w)
                            if self.tokenizer.count_tokens(" ".join(curr_words)) >= self.chunk_size:
                                units.append({"type": "sub_sentence", "text": " ".join(curr_words)})
                                curr_words = []
                        if curr_words:
                            units.append({"type": "sub_sentence", "text": " ".join(curr_words)})
        return units

    def _split_markdown_table_if_needed(self, table_text: str) -> List[str]:
        if self.tokenizer.count_tokens(table_text) <= self.chunk_size:
            return [table_text]
        lines = [l for l in table_text.splitlines() if l.strip()]
        if len(lines) < 3:
            return [table_text]
        header_prefix = f"{lines[0]}\n{lines[1]}\n"
        header_tokens = self.tokenizer.count_tokens(header_prefix)
        table_chunks, current_rows, current_tokens = [], [], header_tokens
        for row in lines[2:]:
            r_tokens = self.tokenizer.count_tokens(row)
            if current_tokens + r_tokens > self.chunk_size and current_rows:
                table_chunks.append(header_prefix + "\n".join(current_rows))
                current_rows, current_tokens = [row], header_tokens + r_tokens
            else:
                current_rows.append(row)
                current_tokens += r_tokens
        if current_rows:
            table_chunks.append(header_prefix + "\n".join(current_rows))
        return table_chunks

    def _extract_overlap_units(self, text_units: List[str]) -> List[str]:
        overlap_units, tokens_acc = [], 0
        for u in reversed(text_units):
            if u.startswith("|") and "\n|" in u:
                continue
            u_tok = self.tokenizer.count_tokens(u)
            if tokens_acc + u_tok <= self.chunk_overlap or not overlap_units:
                overlap_units.insert(0, u)
                tokens_acc += u_tok
            else:
                break
        return overlap_units

    def chunk_document_page(self, page: DocumentPage) -> List[DocumentChunk]:
        cleaned_text = self.cleaner.sanitize(page.content)
        if not cleaned_text:
            return []
        structural_units = self._split_into_structural_units(cleaned_text)
        if not structural_units:
            return []
        chunks, current_unit_texts, current_token_count, current_chapter = [], [], 0, None
        chunk_counter = 0

        def create_chunk(text_parts: List[str], chapter_name: Optional[str], c_idx: int) -> Optional[DocumentChunk]:
            chunk_content = "\n\n".join(text_parts).strip()
            if not chunk_content:
                return None
            t_count = self.tokenizer.count_tokens(chunk_content)
            source_file = page.metadata.get("sumber", page.metadata.get("file_name", "unknown"))
            unique_seed = f"{source_file}_p{page.page_number}_c{c_idx}_{chunk_content[:40]}"
            chunk_hash = hashlib.sha256(unique_seed.encode("utf-8")).hexdigest()[:16]
            meta = dict(page.metadata)
            meta.update({
                "sumber": source_file,
                "tipe": page.metadata.get("tipe", "txt"),
                "halaman": page.page_number,
                "klasifikasi": page.metadata.get("klasifikasi", "secret"),
                "classification": page.metadata.get("classification", "SECRET"),
                "chunk_index": c_idx,
                "token_count": t_count,
                "char_length": len(chunk_content),
                "judul_bab": chapter_name or meta.get("judul_bab", "UMUM"),
            })
            return DocumentChunk(chunk_id=f"doc_{chunk_hash}", text=chunk_content, metadata=meta)

        for unit in structural_units:
            u_text, u_type = unit["text"], unit["type"]
            if u_type == "heading":
                current_chapter = u_text
            if u_type == "table":
                for st in self._split_markdown_table_if_needed(u_text):
                    st_tokens = self.tokenizer.count_tokens(st)
                    if current_token_count + st_tokens > self.chunk_size and current_unit_texts:
                        c = create_chunk(current_unit_texts, current_chapter, chunk_counter)
                        if c:
                            chunks.append(c)
                            chunk_counter += 1
                        current_unit_texts = self._extract_overlap_units(current_unit_texts) + [st]
                        current_token_count = self.tokenizer.count_tokens("\n\n".join(current_unit_texts))
                    else:
                        current_unit_texts.append(st)
                        current_token_count += st_tokens
                continue
            u_tokens = self.tokenizer.count_tokens(u_text)
            if current_token_count + u_tokens > self.chunk_size and current_unit_texts:
                c = create_chunk(current_unit_texts, current_chapter, chunk_counter)
                if c:
                    chunks.append(c)
                    chunk_counter += 1
                current_unit_texts = self._extract_overlap_units(current_unit_texts) + [u_text]
                current_token_count = self.tokenizer.count_tokens("\n\n".join(current_unit_texts))
            else:
                current_unit_texts.append(u_text)
                current_token_count += u_tokens

        if current_unit_texts:
            c = create_chunk(current_unit_texts, current_chapter, chunk_counter)
            if c:
                chunks.append(c)
        return chunks

    def chunk_pages(self, pages: List[DocumentPage]) -> List[DocumentChunk]:
        all_chunks: List[DocumentChunk] = []
        for page in pages:
            all_chunks.extend(self.chunk_document_page(page))
        return all_chunks

# Snapshot terkompresi (LZMA) dari 16 dokumen di folder data/raw/SECRET (34 bagian/lembar kerja)
# Digunakan secara otomatis di VM Google Colab (/content) karena folder lokal laptop tidak ada di server Colab.
_BUNDLED_SECRET_PAGES_LZMA_B64 = (
    "..."
)

def load_secret_pages(data_dir: Path) -> List[DocumentPage]:
    """Memuat halaman dokumen dari bundel data/raw/SECRET yang telah disinkronkan ke notebook."""
    raw_json = lzma.decompress(base64.b64decode(_BUNDLED_SECRET_PAGES_LZMA_B64)).decode("utf-8")
    records = json.loads(raw_json)
    pages = [
        DocumentPage(content=r["content"], page_number=r["page_number"], metadata=r["metadata"])
        for r in records
    ]
    unique_files = sorted({p.metadata.get("sumber", "") for p in pages})
    logger.info(f"Berhasil memuat {len(pages)} bagian/halaman dari {len(unique_files)} dokumen SECRET:")
    for idx, fname in enumerate(unique_files, 1):
        logger.info(f"  [{idx:02d}] {fname}")
    return pages

print("✅ Modul Ingestion, TextCleaner, TextChunker & Bundel 16 Dokumen SECRET siap digunakan!")


✅ Modul Ingestion, TextCleaner, TextChunker & Bundel 16 Dokumen SECRET siap digunakan!


In [20]:
# 4. Definisi Mandiri EmbeddingEngine (Model Nomic Embed v1.5 di GPU CUDA) & ChromaVectorStore
import gc
import chromadb
from chromadb.config import Settings as ChromaSettings
from transformers import PreTrainedModel
from sentence_transformers import SentenceTransformer

# Patch kompatibilitas transformers >= 5.0 untuk NomicBertModel.get_extended_attention_mask
def _nomic_get_extended_attention_mask(self, attention_mask, input_shape, device_or_dtype=None):
    if attention_mask is None:
        return None
    if attention_mask.dim() == 3:
        extended_attention_mask = attention_mask[:, None, :, :]
    elif attention_mask.dim() == 2:
        extended_attention_mask = attention_mask[:, None, None, :]
    else:
        extended_attention_mask = attention_mask
    extended_attention_mask = extended_attention_mask.to(dtype=getattr(self, "dtype", torch.float32))
    return (1.0 - extended_attention_mask) * -10000.0

if not hasattr(PreTrainedModel, "get_extended_attention_mask"):
    PreTrainedModel.get_extended_attention_mask = _nomic_get_extended_attention_mask

class EmbeddingEngine:
    """Mesin Embedding menggunakan model nomic-ai/nomic-embed-text-v1.5 pada GPU CUDA Google Colab."""
    def __init__(self, model_name: str = None, device: str = None, batch_size: int = None):
        emb_cfg = SETTINGS["embedding"]
        self.model_name = model_name or emb_cfg["model_name"]
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = int(batch_size or emb_cfg["batch_size"])
        self.doc_prefix = emb_cfg["document_prefix"]
        self.query_prefix = emb_cfg["query_prefix"]
        self.normalize = emb_cfg["normalize_embeddings"]
        self._model = None

    def _empty_accelerator_cache(self):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def _get_model(self) -> SentenceTransformer:
        if self._model is None:
            logger.info(f"Mengunduh/Memuat model embedding '{self.model_name}' ke device '{self.device}'...")
            self._model = SentenceTransformer(
                self.model_name,
                trust_remote_code=True,
                device=self.device
            )
        return self._model

    def embed_documents(self, documents: List[str]) -> List[List[float]]:
        if not documents:
            return []
        prefixed_texts = [
            f"{self.doc_prefix}{doc}" if "nomic" in self.model_name.lower() and not doc.startswith(self.doc_prefix) else doc
            for doc in documents
        ]
        model = self._get_model()
        logger.info(f"Menjalankan model.encode() untuk {len(prefixed_texts)} chunks pada {self.device.upper()} (batch_size={self.batch_size})...")
        embeddings = model.encode(
            prefixed_texts,
            batch_size=self.batch_size,
            show_progress_bar=True,
            normalize_embeddings=self.normalize
        )
        self._empty_accelerator_cache()
        return embeddings.tolist()

class ChromaVectorStore:
    """Antarmuka penyimpanan vektor ChromaDB (PersistentClient di VM Google Colab)."""
    def __init__(self, persist_directory: str = None, collection_name: str = None):
        cfg = SETTINGS["chromadb"]
        self.persist_directory = persist_directory or cfg["persist_directory"]
        self.collection_name = collection_name or cfg["collection_name"]
        self.distance_function = cfg["distance_function"]
        Path(self.persist_directory).mkdir(parents=True, exist_ok=True)

        self.client = chromadb.PersistentClient(
            path=self.persist_directory,
            settings=ChromaSettings(anonymized_telemetry=False, allow_reset=True)
        )
        self.collection = self.get_or_create_collection(self.collection_name)

    def get_or_create_collection(self, name: str):
        return self.client.get_or_create_collection(
            name=name,
            metadata={"hnsw:space": self.distance_function}
        )

    def upsert_chunks(self, chunks: List[DocumentChunk], embeddings: List[List[float]], batch_size: int = 200) -> int:
        total_upserted = 0
        total_chunks = len(chunks)
        for i in range(0, total_chunks, batch_size):
            b_chunks = chunks[i : i + batch_size]
            b_embs = embeddings[i : i + batch_size]
            ids = [c.chunk_id for c in b_chunks]
            docs = [c.text for c in b_chunks]
            metas = [
                {k: (v if isinstance(v, (str, int, float, bool)) else str(v)) for k, v in c.metadata.items()}
                for c in b_chunks
            ]
            self.collection.upsert(ids=ids, documents=docs, embeddings=b_embs, metadatas=metas)
            total_upserted += len(b_chunks)
            logger.info(f"Tersimpan {total_upserted}/{total_chunks} chunks ke ChromaDB.")
        return total_upserted

    def get_stats(self) -> Dict[str, Any]:
        return {
            "collection_name": self.collection_name,
            "total_vectors": self.collection.count(),
            "persist_directory": self.persist_directory,
        }

print("✅ Class EmbeddingEngine & ChromaVectorStore siap digunakan!")


✅ Class EmbeddingEngine & ChromaVectorStore siap digunakan!


In [21]:
# 5. Jalankan Pipeline build_index pada Dokumen data/raw/SECRET dengan GPU Google Colab
import time
from transformers import PreTrainedModel

# Pastikan patch get_extended_attention_mask terpasang meskipun hanya Sel 5 yang dijalankan ulang
if not hasattr(PreTrainedModel, "get_extended_attention_mask"):
    def _nomic_get_extended_attention_mask(self, attention_mask, input_shape, device_or_dtype=None):
        if attention_mask is None:
            return None
        if attention_mask.dim() == 3:
            extended_attention_mask = attention_mask[:, None, :, :]
        elif attention_mask.dim() == 2:
            extended_attention_mask = attention_mask[:, None, None, :]
        else:
            extended_attention_mask = attention_mask
        extended_attention_mask = extended_attention_mask.to(dtype=getattr(self, "dtype", torch.float32))
        return (1.0 - extended_attention_mask) * -10000.0
    PreTrainedModel.get_extended_attention_mask = _nomic_get_extended_attention_mask

def run_indexing(
    data_dir: Path,
    reset_collection: bool = False,
    device: str = None,
    batch_size: int = None
):
    logger.info("=== Memulai Pipeline Indexing Dokumen PUSDIKZI (GPU Google Colab) ===")
    start_time = time.time()

    # 1. Inisialisasi Vector Store (ChromaDB Persistent di VM Colab)
    vector_store = ChromaVectorStore()
    if reset_collection:
        logger.warning(f"Mereset koleksi '{vector_store.collection_name}'...")
        vector_store.client.delete_collection(vector_store.collection_name)
        vector_store.collection = vector_store.get_or_create_collection(vector_store.collection_name)

    # 2. Muat 16 Dokumen SECRET (34 bagian/lembar kerja)
    pages = load_secret_pages(data_dir)
    logger.info(f"Total bagian/halaman dokumen yang berhasil diekstrak: {len(pages)}")

    # 3. Processing & Structure-Aware Chunking (256 token, 10% overlap)
    chunker = TextChunker(
        chunk_size=SETTINGS["ingestion"]["chunking"]["chunk_size"],
        chunk_overlap=SETTINGS["ingestion"]["chunking"]["chunk_overlap"]
    )
    chunks = chunker.chunk_pages(pages)
    logger.info(f"Total teks terpotong menjadi {len(chunks)} chunks.")

    # 4. Generate Vector Embedding dengan Model di GPU CUDA
    engine = EmbeddingEngine(device=device, batch_size=batch_size)
    texts_to_embed = [c.text for c in chunks]
    t0 = time.time()
    embeddings = engine.embed_documents(texts_to_embed)
    t_emb = time.time() - t0
    logger.info(f"Selesai menghasilkan {len(embeddings)} vector embedding dalam {t_emb:.2f} detik.")

    # 5. Simpan Chunks & Vektor ke ChromaDB
    total_stored = vector_store.upsert_chunks(chunks, embeddings)

    # 6. Laporan Akhir
    elapsed = time.time() - start_time
    stats = vector_store.get_stats()
    logger.info("==================================================")
    logger.info("        LAPORAN INDEXING DATABASE VEKTOR          ")
    logger.info("==================================================")
    logger.info(f"Status              : Sukses")
    logger.info(f"Target Folder       : {data_dir}")
    logger.info(f"Dokumen Pages       : {len(pages)}")
    logger.info(f"Total Chunks        : {total_stored}")
    logger.info(f"Total Vektor di DB  : {stats.get('total_vectors')}")
    logger.info(f"Koleksi ChromaDB    : {stats.get('collection_name')}")
    logger.info(f"Device GPU          : {engine.device}")
    logger.info(f"Waktu Eksekusi      : {elapsed:.2f} detik")
    logger.info("==================================================")
    return vector_store

DATA_DIR = Path("./data/raw/SECRET")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64

vector_store = run_indexing(
    data_dir=DATA_DIR,
    reset_collection=True,
    device=DEVICE,
    batch_size=BATCH_SIZE
)


Batches:   0%|          | 0/87 [00:00<?, ?it/s]

In [22]:
# 6. Ekspor Hasil Vektor dari VM Colab (Unduh File JSON / ZIP ke Laptop Anda)
import json
import shutil
from pathlib import Path

store = ChromaVectorStore()
total_vecs = store.collection.count()
print(f"Total vektor di dalam koleksi '{store.collection_name}': {total_vecs}")

if total_vecs > 0:
    export_dir = Path("./data/processed")
    export_dir.mkdir(parents=True, exist_ok=True)
    export_json_path = export_dir / "colab_indexed_vectors.json"

    all_data = store.collection.get(include=["documents", "metadatas", "embeddings"])
    payload = {
        "collection_name": store.collection_name,
        "total_vectors": total_vecs,
        "ids": all_data["ids"],
        "documents": all_data["documents"],
        "metadatas": all_data["metadatas"],
        "embeddings": [
            emb.tolist() if hasattr(emb, "tolist") else list(emb)
            for emb in all_data["embeddings"]
        ],
    }
    with open(export_json_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False)
    size_mb = export_json_path.stat().st_size / (1024 * 1024)
    print(f"✅ Berhasil mengekspor {total_vecs} vektor ke: {export_json_path.resolve()} ({size_mb:.2f} MB)")

    persist_dir = Path(store.persist_directory)
    if persist_dir.exists():
        zip_file = shutil.make_archive(str(export_dir / "chroma_colab_db"), "zip", str(persist_dir))
        print(f"✅ Arsip database ChromaDB tersimpan di: {zip_file}")


Total vektor di dalam koleksi 'pusdikzi_knowledge_base': 5546
✅ Berhasil mengekspor 5546 vektor ke: /content/data/processed/colab_indexed_vectors.json (97.73 MB)
✅ Arsip database ChromaDB tersimpan di: /content/data/processed/chroma_colab_db.zip


In [ ]:
# 7. [OPSIONAL - JALANKAN DI KERNEL LOKAL LAPTOP] Import Vektor Hasil Colab ke Docker ChromaDB Lokal
import json
import chromadb
from pathlib import Path

export_file = Path("./data/processed/colab_indexed_vectors.json")
if not export_file.exists():
    export_file = Path("../data/processed/colab_indexed_vectors.json")

if export_file.exists():
    with open(export_file, "r", encoding="utf-8") as f:
        bundle = json.load(f)

    local_client = chromadb.HttpClient(host="localhost", port=8000)
    col = local_client.get_or_create_collection(
        name=bundle.get("collection_name", "pusdikzi_knowledge_base"),
        metadata={"hnsw:space": "cosine"}
    )

    ids, docs, metas, embs = bundle["ids"], bundle["documents"], bundle["metadatas"], bundle["embeddings"]
    batch_sz = 200
    for i in range(0, len(ids), batch_sz):
        col.upsert(
            ids=ids[i : i + batch_sz],
            documents=docs[i : i + batch_sz],
            metadatas=metas[i : i + batch_sz],
            embeddings=embs[i : i + batch_sz]
        )
    print(f"✅ Berhasil mengimpor {col.count()} vektor hasil GPU Colab ke Docker ChromaDB lokal (localhost:8000)!")
else:
    print("File colab_indexed_vectors.json belum ditemukan di lokal.")
